# API key usage to try to gather the data of the guardian website.

Goal gather a few hundereds of fresh news articles, with text body in propper format (potentially from a similar source)

In [12]:
import requests
import json
import os
import pandas as pd

In [13]:
API_KEY_GUARDIAN = "77c894e3-36f6-4d1c-99db-9e3fe5693458"
end_point = "https://content.guardianapis.com/search"


In [14]:
# Do a API query to the Guardian API
params = {
    "api-key": API_KEY_GUARDIAN,
    "q": "artificial intelligence",   # search query
    "page-size": 2,
    "show-fields": "headline,byline,bodyText",
}

response = requests.get(end_point, params=params)
response.raise_for_status()

data = response.json()

print(data["response"]["results"][0].keys())
results = data["response"]["results"]

for art in results:
    body = art["fields"]["bodyText"]
    print(art["type"])
    print(art["webPublicationDate"])
    print(art["webTitle"])
    print(body[:500], "...\n")


dict_keys(['id', 'type', 'sectionId', 'sectionName', 'webPublicationDate', 'webTitle', 'webUrl', 'apiUrl', 'fields', 'isHosted', 'pillarId', 'pillarName'])
article
2025-06-11T16:59:22Z
Under attack from artificial intelligence | Letters
Your article about Björn Ulvaeus implies that he disagrees with artists who want to protect their copyright because he believes AI isn’t a “creative threat” (Super Trouper meets supercomputer: AI helping Abba star to write musical, 4 June). In fact, in his role as president of the International Confederation of Societies of Authors and Composers, Ulvaeus has spoken in discussions with the UK government about protection for artists from “profit-seeking tech companies”. In a speech to MPs and peer ...

article
2025-12-06T15:00:55Z
Artificial intelligence research has a slop problem, academics say: ‘It’s a mess’
A single person claims to have authored 113 academic papers on artificial intelligence this year, 89 of which will be presented this week at one o

In [15]:
def fetch_guardian_articles_fresh(page_size=10, page=1, date_from="2025-12-01"):
    """
    Fetch articles from The Guardian API for 2026.
    
    Args:
        page_size: Number of articles per page (default 10)
        page: Page number (default 1)
    
    Returns:
        DataFrame with columns: publication_date, title, topic, body
    """
    params = {
        "api-key": API_KEY_GUARDIAN,
        "q": "",
        "from-date": date_from,
        # "to-date": "2026-12-31",
        "page-size": page_size,
        "page": page,
        "show-fields": "headline,byline,bodyText",
    }
    
    response = requests.get(end_point, params=params)
    response.raise_for_status()
    
    data = response.json()
    results = data["response"]["results"]
    
    articles_list = []
    for art in results:
        articles_list.append({
            "publication_date": art["webPublicationDate"],
            "title": art["webTitle"],
            "topic": art["sectionName"],
            "body": art["fields"]["bodyText"]
        })
    
    return articles_list

In [17]:
# Fetch articles
total_articles = []
for page_num in range(1, 50):  # 
    articles_df = fetch_guardian_articles_fresh(page_size=200, page=page_num, date_from="2025-11-01")
    total_articles.extend(articles_df)
    print("Fetched articles_df for page", page_num, "number:", len(articles_df))

# Convert to DataFrame
df_guardian_2026 = pd.DataFrame(total_articles)

Fetched articles_df for page 1 number: 200
Fetched articles_df for page 2 number: 200
Fetched articles_df for page 3 number: 200
Fetched articles_df for page 4 number: 200
Fetched articles_df for page 5 number: 200
Fetched articles_df for page 6 number: 200
Fetched articles_df for page 7 number: 200
Fetched articles_df for page 8 number: 200
Fetched articles_df for page 9 number: 200
Fetched articles_df for page 10 number: 200
Fetched articles_df for page 11 number: 200
Fetched articles_df for page 12 number: 200
Fetched articles_df for page 13 number: 200
Fetched articles_df for page 14 number: 200
Fetched articles_df for page 15 number: 200
Fetched articles_df for page 16 number: 200
Fetched articles_df for page 17 number: 200
Fetched articles_df for page 18 number: 200
Fetched articles_df for page 19 number: 200
Fetched articles_df for page 20 number: 200
Fetched articles_df for page 21 number: 200
Fetched articles_df for page 22 number: 200
Fetched articles_df for page 23 number: 2

In [20]:
# Display the DataFrame
print(df_guardian_2026["body"].iloc[0])
print(df_guardian_2026["body"].iloc[200])
print(df_guardian_2026["body"].iloc[400])
print(df_guardian_2026["body"].iloc[599])
print(df_guardian_2026["body"].iloc[600])
print(df_guardian_2026["body"].iloc[601])
print(df_guardian_2026["body"].iloc[799])


The former editor of the Daily Mail, Paul Dacre, is to be called as a witness in the legal action brought by the Duke of Sussex and six other household names against the newspaper’s publishers over allegations of unlawful information gathering, the high court was told. Antony White KC, for Associated Newspapers Limited (ANL), said Dacre, 77, now the editor-in-chief of ANL’s DMG Media company, and Peter Wright, a former editor of the Mail on Sunday, could be called as early defence witnesses in the trial, scheduled to begin on 19 January. It was “critically important for various reasons that Mr Dacre and Mr Wright are able to go out over the top first” to deal with “critically important” allegations “before they send their troops into battle”, White told the judge, Mr Justice Nicklin, at a pre-trial hearing. David Sherborne, for the claimants, indicated that ANL wanted to call Dacre first in relation to “evidence he gave to the inquiry”, referring to the 2011-12 Leveson inquiry into pr

In [21]:
def save_text_bodies_to_files(df, file_path):    
    with open(file_path, 'w', encoding='utf-8') as f:
        for i, row in df.iterrows():
            f.write(row['body']+"\n")

In [22]:
out_file = "../text_data/guardian_from_nov2025_articles_v1.10k"
save_text_bodies_to_files(df_guardian_2026, out_file)